In [ ]:
#Now let's add simple, cheap features to the data
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

def get_combined_features(text):
    if not isinstance(text, str) or text.strip() == "":
        return [0] * 13 #Returns 0s for all 13 features

    #1. TextBlob for Sentiment/Trustworthiness
    blob = TextBlob(text)
    polarity = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity

    #2. SpaCy for Linguistic & POS features
    doc = nlp(text)
    tokens = [t for t in doc if not t.is_space]
    words = [t for t in tokens if not t.is_punct]

    #Basic Counts
    char_count = len(text)
    word_count = len(words)
    total_tokens = len(tokens)
    avg_word_len = np.mean([len(w.text) for w in words]) if words else 0

    #Stylometry (Propaganda Signals)
    caps_count = sum(1 for w in words if w.text.isupper() and len(w.text) > 1)
    title_count = sum(1 for w in words if w.text.istitle())
    punct_count = sum(1 for t in tokens if any(c in '!?."' for c in t.text))
    sent_count = len(re.split(r'[.!?]+', text))

    #POS Tagging (Adjectives & Adverbs)
    adj_count = sum(1 for t in doc if t.pos_ == "ADJ")
    adv_count = sum(1 for t in doc if t.pos_ == "ADV")

    #Lexical Diversity (Type-Token Ratio)
    unique_words = len(set([w.text.lower() for w in words]))
    ttr = unique_words / max(word_count, 1)

    #3. Normalization
    caps_ratio = caps_count / max(word_count, 1)
    punct_ratio = punct_count / max(word_count, 1)
    adj_density = adj_count / max(total_tokens, 1)
    adv_density = adv_count / max(total_tokens, 1)

    return [
        char_count, word_count, avg_word_len, caps_ratio, punct_ratio,
        sent_count, title_count, total_tokens,
        polarity, subjectivity, adj_density, adv_density, ttr
    ]

In [ ]:
feature_cols = ['char_count', 'word_count', 'avg_word_len', 'caps_ratio', 'punct_ratio','sent_count', 'title_count', 'total_tokens', 'polarity', 'subjectivity', 'adj_density', 'adv_density', 'lexical_diversity']

if second_output_file.exists():
    df = pd.read_csv(second_output_file)
    df.head()
else:
    print("Getting features for all articles (this will take a while)...")
    tqdm.pandas()
    features = news['text'].progress_apply(get_combined_features).tolist()

    print("Saving DataFrame...")
    features_df = pd.DataFrame(features, columns=feature_cols)
    df = pd.concat([news, features_df], axis=1)
    df.to_csv(second_output_file, index=False)
    df.head()

In [ ]:
#Train/test split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
#Add vectorized word counts, limit to 1000 features so the resulting CSV isn't gigabytes in size
tfidf = TfidfVectorizer(max_features=1000, stop_words='english')

#Fit on training data only to avoid data leakage
X_train_tfidf = tfidf.fit_transform(train_df['text'].fillna(""))
X_test_tfidf = tfidf.transform(test_df['text'].fillna(""))

In [ ]:
#Convert TF-IDF sparse matrix to a DataFrame
tfidf_cols = [f"word_{name}" for name in tfidf.get_feature_names_out()]
train_tfidf_df = pd.DataFrame(X_train_tfidf.toarray(), columns=tfidf_cols, index=train_df.index)
test_tfidf_df = pd.DataFrame(X_test_tfidf.toarray(), columns=tfidf_cols, index=test_df.index)

In [ ]:
#We use the specialist training data (only rows with propaganda) to find
#which words are most unique to each specific technique.
def get_top_keywords_per_class(X_tfidf, y_spec, tfidf_vectorizer, n_words=15):
    feature_names = tfidf_vectorizer.get_feature_names_out()
    class_keywords = {}

    for i, class_name in enumerate(all_techniques):
        #Chi2 finds words with the highest correlation to this class
        _, pval = chi2(X_tfidf, y_spec[:, i])
        top_indices = np.argsort(pval)[:n_words]
        class_keywords[class_name] = [feature_names[idx] for idx in top_indices]
    return class_keywords

In [ ]:
y_train_binary = (train_df['propaganda'].apply(lambda x: len(get_tech_list(x))) > 0).astype(int)
train_has_prop = (y_train_binary == 1)

#Do the same for test data so Stage 2 can be evaluated later
y_test_binary = (test_df['propaganda'].apply(lambda x: len(get_tech_list(x))) > 0).astype(int)
test_has_prop = (y_test_binary == 1)

In [ ]:
mlb = MultiLabelBinarizer()
y_spec_temp = mlb.fit_transform(train_df[train_has_prop]['propaganda'].apply(get_tech_list))
X_spec_tfidf_temp = X_train_tfidf[train_has_prop.values]

In [ ]:
keywords = get_top_keywords_per_class(X_spec_tfidf_temp, y_spec_temp, tfidf)
print(keywords)